# mervalBotV2 — Backtest estrategia ADR Spread

Objetivo: validar si la estrategia A (ADR Spread) genera ganancia neta con 2 años de datos históricos, descontando comisión.

**Criterio go/no-go para LIVE:** Sharpe > 1.0 después de comisión. Menos que eso, no se opera.

**Arquitectura:** este notebook importa `calc_adr_signal` directo del repo para evitar drift entre backtest y bot en vivo.

**Limitación honesta:** yfinance da 60 días de datos intradía 5m. Para 2 años usamos datos diarios OHLC y asumimos conservadoramente que si intraday toca SL y TP en el mismo día, primero pega el SL (peor caso).

## 1. Instalar dependencias y clonar repo

In [ ]:
!pip install -q yfinance pandas numpy matplotlib

# CONFIGURAR: reemplazar TU_USUARIO por tu usuario de GitHub
REPO_URL = 'https://github.com/TU_USUARIO/mervalBotV2.git'

!rm -rf mervalBotV2
!git clone $REPO_URL

import sys
sys.path.insert(0, '/content/mervalBotV2')

## 2. Importar la función de señal del repo

Si esto falla, el repo no está bien clonado o el refactor de `calc_adr_signal` no está aplicado.

In [ ]:
from strategies.adr_spread import (
    calc_adr_signal,
    ADRSignalResult,
    COMMISSION_RT,
    SLIPPAGE_BUFFER,
    MIN_EDGE,
    MIN_SPREAD_ENTRY,
    SL_PCT,
)

print(f'COMMISSION_RT   = {COMMISSION_RT:.2%}')
print(f'SLIPPAGE_BUFFER = {SLIPPAGE_BUFFER:.2%}')
print(f'MIN_EDGE        = {MIN_EDGE:.2%}')
print(f'MIN_SPREAD_ENTRY= {MIN_SPREAD_ENTRY:.2%}')
print(f'SL_PCT          = {SL_PCT:.2%}')

# Smoke test
test = calc_adr_signal(adr_usd=25.0, mep=1000.0, byma_last=22000.0, ratio=10)
assert test is not None, 'calc_adr_signal importado mal'
print(f'\nSmoke test OK: spread={test.spread*100:.2f}%, signal={test.should_signal}')

## 3. Configuración del backtest

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

PAIRS = [
    {'byma': 'GGAL.BA', 'adr': 'GGAL', 'ratio': 10, 'label': 'GGAL'},
    {'byma': 'YPFD.BA', 'adr': 'YPF',  'ratio': 1,  'label': 'YPFD'},
    {'byma': 'PAMP.BA', 'adr': 'PAM',  'ratio': 25, 'label': 'PAMP'},
    {'byma': 'BMA.BA',  'adr': 'BMA',  'ratio': 10, 'label': 'BMA'},
]

START = '2024-01-01'
END   = '2026-04-01'

INITIAL_CAPITAL_ARS = 500_000
RISK_PER_TRADE     = 0.02   # 2% capital por trade
MAX_POSITION_PCT   = 0.25   # 25% capital máximo por trade

# Fuente MEP para backtest — pragmático:
# Usamos el par GGAL.BA / GGAL como MEP implícito diario.
# Limitación: el día que la señal es sobre GGAL, el MEP está perfectamente
# sincronizado con GGAL, lo que puede sesgar resultados a la baja en ese ticker.
# Para los otros 3 tickers es independiente.
MEP_SOURCE = {'byma': 'GGAL.BA', 'adr': 'GGAL', 'ratio': 10}

## 4. Descargar datos históricos

In [ ]:
def dl(symbol, start, end):
    df = yf.download(symbol, start=start, end=end, progress=False, auto_adjust=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df[['Open', 'High', 'Low', 'Close']].dropna()

all_symbols = set()
for p in PAIRS:
    all_symbols.add(p['byma'])
    all_symbols.add(p['adr'])
all_symbols.add(MEP_SOURCE['byma'])
all_symbols.add(MEP_SOURCE['adr'])

data = {}
for s in sorted(all_symbols):
    df = dl(s, START, END)
    print(f'{s:12s}: {len(df):4d} días | {df.index.min().date()} → {df.index.max().date()}')
    data[s] = df

## 5. Construir serie de MEP implícito

MEP = precio_BYMA × ratio / precio_ADR_USD (en el par de referencia)

In [ ]:
mep_byma = data[MEP_SOURCE['byma']]['Close']
mep_adr  = data[MEP_SOURCE['adr']]['Close']
mep_df = pd.concat([mep_byma, mep_adr], axis=1, keys=['byma', 'adr']).dropna()
mep_df['mep'] = mep_df['byma'] * MEP_SOURCE['ratio'] / mep_df['adr']
mep_series = mep_df['mep']

print(f'MEP range: {mep_series.min():.2f} → {mep_series.max():.2f}')
print(f'MEP mediana: {mep_series.median():.2f}')

mep_series.plot(figsize=(12, 3), title='MEP implícito (ARS/USD)')
plt.grid(True); plt.show()

## 6. Simulación día por día

Para cada día D:
1. Tomo cierre de BYMA del día D-1 como `byma_last` (proxy del precio pre-open de D)
2. Tomo cierre del ADR del día D-1 como `adr_usd`
3. Tomo MEP del día D-1
4. Llamo a `calc_adr_signal`
5. Si hay señal BUY, simulo entrada al **Open** de BYMA del día D
6. Simulo SL/TP con High/Low del día D (asunción pesimista: si High≥TP y Low≤SL, pega SL primero)
7. Si no toca ninguno, exit al Close del día D (proxy del exit forzado)

In [ ]:
def simulate_day(byma_row, adr_prev_close, mep_prev, byma_prev_close, ratio, capital):
    """Retorna dict con resultado del día o None si no hubo señal."""
    result = calc_adr_signal(
        adr_usd=adr_prev_close,
        mep=mep_prev,
        byma_last=byma_prev_close,
        ratio=ratio,
    )
    if result is None or not result.should_signal:
        return None

    # Entrada al open del día D (no al close anterior)
    entry = byma_row['Open']
    sl = entry * (1 - SL_PCT)
    # TP recalculado con entry real (no con close anterior)
    net_target = max(result.spread - COMMISSION_RT, 0.01)
    tp = entry * (1 + net_target)

    high = byma_row['High']
    low  = byma_row['Low']
    close = byma_row['Close']

    # Decisión pesimista: si ambos toca, asumo SL primero
    if low <= sl and high >= tp:
        exit_px, exit_reason = sl, 'SL (both hit, pesimista)'
    elif low <= sl:
        exit_px, exit_reason = sl, 'SL'
    elif high >= tp:
        exit_px, exit_reason = tp, 'TP'
    else:
        exit_px, exit_reason = close, 'EOD'

    # Sizing
    risk_ars = capital * RISK_PER_TRADE
    risk_per_share = entry - sl
    qty_by_risk = int(risk_ars / risk_per_share) if risk_per_share > 0 else 0
    qty_by_cap  = int((capital * MAX_POSITION_PCT) // entry)
    qty = min(qty_by_risk, qty_by_cap)
    if qty <= 0:
        return None

    gross_pnl = (exit_px - entry) * qty
    commission = (entry + exit_px) * qty * (COMMISSION_RT / 2)
    net_pnl = gross_pnl - commission

    return {
        'entry': entry, 'sl': sl, 'tp': tp, 'exit': exit_px,
        'exit_reason': exit_reason, 'qty': qty,
        'gross_pnl': gross_pnl, 'commission': commission, 'net_pnl': net_pnl,
        'pnl_pct_net': net_pnl / (entry * qty),
        'spread_signal': result.spread,
    }


def run_backtest_one_ticker(pair, capital):
    byma_df = data[pair['byma']]
    adr_df  = data[pair['adr']]
    aligned = byma_df.join(adr_df[['Close']].rename(columns={'Close': 'adr_close'}), how='inner')
    aligned = aligned.join(mep_series.rename('mep'), how='inner').dropna()

    trades = []
    equity = capital
    for i in range(1, len(aligned)):
        today = aligned.iloc[i]
        yesterday = aligned.iloc[i - 1]
        res = simulate_day(
            byma_row=today,
            adr_prev_close=yesterday['adr_close'],
            mep_prev=yesterday['mep'],
            byma_prev_close=yesterday['Close'],
            ratio=pair['ratio'],
            capital=equity,
        )
        if res is None:
            continue
        res['date'] = aligned.index[i]
        res['ticker'] = pair['label']
        equity += res['net_pnl']
        res['equity_after'] = equity
        trades.append(res)

    return pd.DataFrame(trades)


results = {}
for pair in PAIRS:
    print(f'Simulando {pair["label"]}...')
    results[pair['label']] = run_backtest_one_ticker(pair, INITIAL_CAPITAL_ARS)
    print(f'  {len(results[pair["label"]])} trades')

## 7. Métricas

In [ ]:
def compute_metrics(trades_df, initial_capital):
    if trades_df.empty:
        return {'n_trades': 0, 'note': 'sin señales'}
    n = len(trades_df)
    wins = (trades_df['net_pnl'] > 0).sum()
    total_pnl = trades_df['net_pnl'].sum()
    returns = trades_df['pnl_pct_net']
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    equity = initial_capital + trades_df['net_pnl'].cumsum()
    peak = equity.cummax()
    dd = (equity - peak) / peak
    max_dd = dd.min()
    return {
        'n_trades': n,
        'win_rate': wins / n,
        'avg_pnl_pct': returns.mean() * 100,
        'total_pnl_ars': total_pnl,
        'total_return_pct': total_pnl / initial_capital * 100,
        'sharpe': sharpe,
        'max_dd_pct': max_dd * 100,
        'sl_hits': (trades_df['exit_reason'].str.startswith('SL')).sum(),
        'tp_hits': (trades_df['exit_reason'] == 'TP').sum(),
        'eod_hits': (trades_df['exit_reason'] == 'EOD').sum(),
    }

rows = []
for label, df in results.items():
    m = compute_metrics(df, INITIAL_CAPITAL_ARS)
    m['ticker'] = label
    rows.append(m)

metrics_df = pd.DataFrame(rows).set_index('ticker')
metrics_df

## 8. Portfolio combinado (todos los tickers, capital compartido)

In [ ]:
all_trades = pd.concat(results.values()).sort_values('date').reset_index(drop=True)
equity_curve = INITIAL_CAPITAL_ARS + all_trades['net_pnl'].cumsum()

portfolio_metrics = compute_metrics(all_trades, INITIAL_CAPITAL_ARS)
print('=== Portfolio combinado ===')
for k, v in portfolio_metrics.items():
    print(f'  {k:20s}: {v}')

print('\n=== Go/No-Go ===')
if portfolio_metrics.get('sharpe', 0) > 1.0:
    print('  ✅ Sharpe > 1.0: candidato a LIVE (revisar también max DD y # trades)')
else:
    print('  ❌ Sharpe <= 1.0: NO operar live. Buscar otra edge.')

## 9. Gráficos

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(all_trades['date'], equity_curve.values)
axes[0].axhline(INITIAL_CAPITAL_ARS, color='gray', ls='--', alpha=0.5)
axes[0].set_title(f'Equity curve (inicio: ${INITIAL_CAPITAL_ARS:,.0f} ARS)')
axes[0].grid(True)

peak = equity_curve.cummax()
dd = (equity_curve - peak) / peak * 100
axes[1].fill_between(all_trades['date'], dd.values, 0, color='red', alpha=0.3)
axes[1].set_title('Drawdown (%)')
axes[1].grid(True)

axes[2].hist(all_trades['pnl_pct_net'] * 100, bins=50, edgecolor='black')
axes[2].axvline(0, color='red', ls='--')
axes[2].set_title('Distribución PnL por trade (%)')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 10. Sanity checks y muestra de trades

In [ ]:
print('=== Primeros 10 trades ===')
print(all_trades.head(10)[['date', 'ticker', 'entry', 'exit', 'exit_reason', 'qty', 'net_pnl', 'pnl_pct_net']].to_string())

print('\n=== Peores 5 trades ===')
print(all_trades.nsmallest(5, 'net_pnl')[['date', 'ticker', 'entry', 'exit', 'exit_reason', 'net_pnl']].to_string())

print('\n=== Mejores 5 trades ===')
print(all_trades.nlargest(5, 'net_pnl')[['date', 'ticker', 'entry', 'exit', 'exit_reason', 'net_pnl']].to_string())

print('\n=== Frecuencia de señales ===')
print(f'Días con trade: {len(all_trades)}')
print(f'Señales por ticker:')
print(all_trades['ticker'].value_counts())

## 11. Análisis de sensibilidad — umbral de entrada

Qué pasaría con distintos valores de `MIN_SPREAD_ENTRY`. Si el óptimo está muy lejos del 2.5% actual, tenemos que tunear el bot.

In [ ]:
def backtest_with_threshold(min_edge_override):
    sens_results = {}
    for pair in PAIRS:
        byma_df = data[pair['byma']]
        adr_df  = data[pair['adr']]
        aligned = byma_df.join(adr_df[['Close']].rename(columns={'Close': 'adr_close'}), how='inner')
        aligned = aligned.join(mep_series.rename('mep'), how='inner').dropna()

        trades = []
        equity = INITIAL_CAPITAL_ARS
        for i in range(1, len(aligned)):
            today = aligned.iloc[i]; yesterday = aligned.iloc[i - 1]
            result = calc_adr_signal(
                adr_usd=yesterday['adr_close'], mep=yesterday['mep'],
                byma_last=yesterday['Close'], ratio=pair['ratio'],
                min_edge=min_edge_override,
            )
            if result is None or not result.should_signal:
                continue
            entry = today['Open']
            sl = entry * (1 - SL_PCT)
            tp = entry * (1 + max(result.spread - COMMISSION_RT, 0.01))
            low, high, close = today['Low'], today['High'], today['Close']
            if low <= sl and high >= tp: exit_px = sl
            elif low <= sl: exit_px = sl
            elif high >= tp: exit_px = tp
            else: exit_px = close
            risk_per_share = entry - sl
            qty = min(
                int((equity * RISK_PER_TRADE) / risk_per_share) if risk_per_share > 0 else 0,
                int((equity * MAX_POSITION_PCT) // entry),
            )
            if qty <= 0: continue
            pnl = (exit_px - entry) * qty - (entry + exit_px) * qty * (COMMISSION_RT / 2)
            equity += pnl
            trades.append({'pnl': pnl, 'pct': pnl / (entry * qty)})
        sens_results[pair['label']] = trades
    all_t = [t for trades in sens_results.values() for t in trades]
    if not all_t:
        return {'threshold': min_edge_override, 'n_trades': 0}
    rets = pd.Series([t['pct'] for t in all_t])
    total = sum(t['pnl'] for t in all_t)
    return {
        'min_edge': min_edge_override,
        'threshold_total': COMMISSION_RT + SLIPPAGE_BUFFER + min_edge_override,
        'n_trades': len(all_t),
        'total_pnl': total,
        'return_pct': total / INITIAL_CAPITAL_ARS * 100,
        'win_rate': (rets > 0).mean(),
        'sharpe': (rets.mean() / rets.std() * np.sqrt(252)) if rets.std() > 0 else 0,
    }

sens = pd.DataFrame([
    backtest_with_threshold(e) for e in [0.003, 0.005, 0.008, 0.012, 0.017, 0.025, 0.035]
])
sens

## 12. Conclusión

- Si **Sharpe > 1.0** en el portfolio combinado Y hay suficientes trades (>30) → candidato a LIVE con capital pequeño
- Si **Sharpe entre 0.5 y 1.0** → edge marginal, posiblemente no sobrevive slippage real. Re-evaluar después de tunear umbral
- Si **Sharpe < 0.5** → la estrategia no funciona, ir a buscar otra cosa
- El análisis de sensibilidad te dice si el umbral actual (2.5%) está cerca del óptimo o hay que cambiarlo en `strategies/adr_spread.py`

**Limitaciones conocidas:**
1. Usamos datos diarios OHLC — no capturamos ejecución exacta intraday
2. Asumimos SL antes que TP cuando el día toca ambos (pesimista)
3. MEP implícito desde GGAL.BA/GGAL — posible sesgo en resultados de GGAL
4. No hay slippage modelado explícitamente (el buffer de 0.5% lo aproxima)
5. Sin survivorship bias check — los 4 tickers son liquidos hoy, pero la tesis asume que lo fueron en todo el período